# Powerflow sparse vs dense Jacobian

DPsim's Newton-Raphson powerflow solver has two implementations selectable on the `Simulation`:

- `PFSolverPowerPolar` (dense Jacobian, a fresh `Eigen::SparseLU` factorization every iteration)
- `PFSolverPowerPolarSparse` (sparse Jacobian with a fixed pattern, symbolic factorization analyzed once and refactorized each iteration)

The dense solver is the default; the sparse solver is opt-in, selected with `set_pf_solver_use_sparse(True/False)` (and only takes effect when DPsim is built with a sparse linear solver). This notebook runs a range of network sizes both ways, checks that the converged voltages are identical, and compares the run time.

In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dpsimpy

In [ ]:
base_url = "https://raw.githubusercontent.com/dpsim-simulator/cim-grid-data/master/Matpower_cases/"
cases = ["case9", "case14"]
for case in cases:
    filename = case + ".xml"
    if not os.path.exists(filename):
        with open(filename, "wb") as out_file:
            out_file.write(requests.get(base_url + case + ".xml", stream=True).content)

In [ ]:
def run_pf(case, use_sparse, final_time=10.0, time_step=1.0):
    filename = case + ".xml"
    name = case + ("_sparse" if use_sparse else "_dense")
    reader = dpsimpy.CIMReader(name)
    system = reader.loadCIM(
        50,
        [filename],
        dpsimpy.Domain.SP,
        dpsimpy.PhaseType.Single,
        dpsimpy.GeneratorType.PVNode,
    )

    sim = dpsimpy.Simulation(name, dpsimpy.LogLevel.off)
    sim.set_system(system)
    sim.set_domain(dpsimpy.Domain.SP)
    sim.set_solver(dpsimpy.Solver.NRP)
    sim.set_pf_solver_use_sparse(use_sparse)
    sim.set_pf_keep_last_solution(False)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)

    t0 = time.perf_counter()
    sim.run()
    runtime = time.perf_counter() - t0

    v_mag = np.array([np.abs(node.attr("v").get()[0][0]) for node in system.nodes])
    return v_mag, runtime

In [ ]:
rows = []
voltage_profiles = {}
for case in cases:
    v_dense, t_dense = run_pf(case, use_sparse=False)
    v_sparse, t_sparse = run_pf(case, use_sparse=True)
    rel_diff = np.max(np.abs(v_sparse - v_dense)) / np.max(np.abs(v_dense))
    rows.append(
        {
            "case": case,
            "buses": len(v_sparse),
            "dense [s]": t_dense,
            "sparse [s]": t_sparse,
            "speedup": t_dense / t_sparse,
            "rel. diff": rel_diff,
        }
    )
    voltage_profiles[case] = (v_sparse, v_dense)

results = pd.DataFrame(rows).set_index("case")
results

In [ ]:
# Both solvers must converge to the same voltages (round-off level)
assert results["rel. diff"].max() < 1e-8

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(results["buses"], results["dense [s]"], "o-", label="dense")
ax1.plot(results["buses"], results["sparse [s]"], "s-", label="sparse")
ax1.set_xlabel("buses")
ax1.set_ylabel("run time [s]")
ax1.set_title("Powerflow run time vs network size")
ax1.legend()

v_sparse, v_dense = voltage_profiles["case14"]
ax2.plot(v_sparse, label="sparse")
ax2.plot(v_dense, "--", label="dense")
ax2.set_xlabel("bus index")
ax2.set_ylabel("voltage magnitude [V]")
ax2.set_title("Voltage profile (case14)")
ax2.legend()

plt.tight_layout()
plt.show()